# MiniGPT — From Tokens to Language
Educational notebook. Run from the repository root. Install requirements and provide the corpus first. Full training may take hours. Attribution: see NOTICE and LICENSE.

In [8]:
import torch
import tiktoken
from model import MiniGPT
from config import MODEL_CONFIG, SEED
from train import train
from generate import generate
torch.manual_seed(SEED)
model = MiniGPT(**MODEL_CONFIG)
print(sum(p.numel() for p in model.parameters()))

6537088


In [9]:
dir(model)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',
 '_named_members',
 '_non_persistent_buffers_se

In [10]:
print(model)

MiniGPT(
  (token_emb): Embedding(50257, 64)
  (pos_emb): Embedding(64, 64)
  (blocks): Sequential(
    (0): TransformerBlock(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (att): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
      )
    )
    (1): TransformerBlock(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (att): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
     

In [7]:
tokenizer = tiktoken.get_encoding("gpt2")
ids = tokenizer.encode("Kolarz rozpoczął wyścig.")
with torch.no_grad():
    logits = model(torch.tensor([ids]))
print(logits.shape)
print(repr(tokenizer.decode([logits[0, -1].argmax().item()])))

torch.Size([1, 17, 50257])
' Jong'


## Train one additional epoch
Run the next cell once per session. It automatically resumes if a checkpoint exists. Wait for the checkpoint confirmation before shutting down. Restarting an interrupted epoch repeats that epoch.

In [ ]:
from pathlib import Path
checkpoint_path = Path("checkpoints/minigpt_checkpoint.pth")
checkpoint = train("data/w_pustyni_i_w_puszczy.txt", epochs=1, resume=checkpoint_path.exists())

In [ ]:
import matplotlib.pyplot as plt
epochs = range(1, len(checkpoint["train_losses"])+1)
plt.plot(epochs, checkpoint["train_losses"], marker="o", label="Train loss")
plt.plot(epochs, checkpoint["val_losses"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.legend()
plt.show()

In [ ]:
print(generate("checkpoints/minigpt_checkpoint.pth", "Staś zobaczył Nel", 40))